In [ ]:
!pip install pandas pytz openpyxl weasyprint pillow jinja2 ipywidgets pypdf

In [ ]:
import pandas as pd
from __future__ import annotations
import logging
import pytz
from openpyxl.worksheet.table import Table, TableStyleInfo
from weasyprint import HTML, CSS
from datetime import datetime, timedelta
from pathlib import Path
import weasyprint
import re
from openpyxl import load_workbook
import locale
from openpyxl.utils import get_column_letter
from dataclasses import dataclass
import os
from openpyxl.styles import NamedStyle
from jinja2 import Environment, FileSystemLoader
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output
import json

## Configuração do Relatório

Escolha o período e o tipo de análise abaixo.

In [ ]:
# Widgets de seleção — escolha os valores e execute a célula seguinte
try:
    ano_atual = datetime.now().year
    tipo_widget = widgets.RadioButtons(
        options=[('Nacional (RD)', 'nacional'), ('Internacional', 'internacional')],
        value='nacional',
        description='Tipo:'
    )

    mes_widget = widgets.Dropdown(
        options=[('Janeiro', 1), ('Fevereiro', 2), ('Março', 3), ('Abril', 4),
                 ('Maio', 5), ('Junho', 6), ('Julho', 7), ('Agosto', 8),
                 ('Setembro', 9), ('Outubro', 10), ('Novembro', 11), ('Dezembro', 12)],
        value=datetime.now().month,
        description='Mês:'
    )

    ano_widget = widgets.BoundedIntText(
        value=ano_atual,
        min=2020,
        max=ano_atual + 1,
        step=1,
        description='Ano:'
    )

    display(tipo_widget, mes_widget, ano_widget)
    WIDGETS_OK = True
except Exception as e:
    print(f"Widgets não disponíveis neste ambiente ({e}).")
    print("Defina manualmente as variáveis TIPO, MES e ANO na célula seguinte.")
    WIDGETS_OK = False

In [ ]:
# ==================== CONFIGURAÇÃO ====================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

try:
    locale.setlocale(locale.LC_ALL, 'pt_BR.UTF-8')
except locale.Error:
    try:
        locale.setlocale(locale.LC_ALL, 'pt_BR')
    except locale.Error:
        logging.warning("Locale pt_BR não disponível neste sistema. Usando padrão.")

CATEGORIAS = {
    "Uber": ["uber"],
    "Estacionamento": ["park", "estacionamento", "Indigo"],
    "Posto": ["posto", "shell", "ipiranga", "automotivo", "Combustive"],
    "Bagagem": ["gol "],
    "Cartão": ["ANUIDADE"],
    "Refeição": ["giraffas", "outback", "lanchonete", "restaurante", "steak",
                 "TCHANBURGUER", "MOEMA NATURAL", "grill", "PARMEGGIO",
                 "MC DONALDS", "CHURRASCARIA", "BAR"]
}

EXPECTED_COLUMNS = [
    'Data da transação', 'Lançamentos', 'Valor', 'Moeda',
    'Valor (US$)', 'Cotação/Dia (R$)', 'Valor em R$', 'Despesa no Exterior'
]

BASE_DIR = Path(r'C:/Users/vital/OneDrive/Documentos/Trabalho/Despesas')

def load_category_memory(path="historico_categorias.json"):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def save_category_memory(memory, path="historico_categorias.json"):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(memory, f, ensure_ascii=False, indent=2)

memory = load_category_memory()

@dataclass
class Transacao:
    Data: datetime
    Lançamento: str
    Valor: float
    Categoria: str
    Observações: str | None = None

def get_latest_invoice_file(folder: Path, mes: str) -> Path | None:
    pattern = f"Fatura_{mes}*.xlsx"
    files = list(folder.glob(pattern))
    if not files:
        return None
    return max(files, key=lambda f: f.stat().st_mtime)

def find_header_row(file_path: Path) -> int | None:
    df = pd.read_excel(file_path, header=None)
    for i, row in df.iterrows():
        if all(col in row.values for col in EXPECTED_COLUMNS):
            return i
    return None

def categorize(descricao: str) -> str:
    desc = descricao.strip().upper()
    if desc in memory:
        return memory[desc]
    for categoria, padroes in CATEGORIAS.items():
        if any(re.search(p.lower(), desc.lower()) for p in padroes):
            return categoria
    return "Categorizar"

def processar(tipo: str, mes: int, ano: int):
    # data_analise aponta para o MÊS SEGUINTE ao selecionado,
    # assim passado (data_analise - 28d) cai dentro do mês desejado
    if mes == 12:
        data_analise = datetime(ano + 1, 1, 1, tzinfo=pytz.timezone("America/Sao_Paulo"))
    else:
        data_analise = datetime(ano, mes + 1, 1, tzinfo=pytz.timezone("America/Sao_Paulo"))
    passado = data_analise - timedelta(days=28)
    futuro = data_analise + timedelta(days=30)
    is_nacional = tipo == 'nacional'
    base = BASE_DIR if is_nacional else BASE_DIR / "Internacional"
    pasta_mes = base / passado.strftime('%y-%m-%b').title()

    if not pasta_mes.exists():
        (pasta_mes / "NF").mkdir(parents=True)
        logging.info(f"Pasta {pasta_mes} criada!")
    else:
        logging.info(f"Pasta {pasta_mes} já existe.")

    meses_nomes = [data_analise.strftime('%B').title(), futuro.strftime('%B').title()]
    transacoes: list[Transacao] = []

    for mes_nome in meses_nomes:
        file_path = get_latest_invoice_file(pasta_mes, mes_nome) or pasta_mes / f"Fatura_{mes_nome}.xlsx"
        if not file_path.exists():
            logging.warning(f"Nenhuma fatura encontrada para {mes_nome}.")
            continue

        try:
            header_row = find_header_row(file_path)
            if header_row is None:
                logging.warning(f"Cabeçalho não encontrado em {mes_nome}.")
                continue

            df = pd.read_excel(file_path, header=header_row)
            df = df.drop(columns=['Unnamed: 0'], errors='ignore')
            df = df.dropna(subset=['Moeda'])
            df['Data da transação'] = pd.to_datetime(df['Data da transação'], errors='coerce')
            df = df.astype({'Valor': 'float32'})
            df = df[df['Despesa no Exterior'] == ("Não" if is_nacional else "Sim")]
            df = df[df['Data da transação'].dt.month == passado.month]

            if is_nacional:
                df = df.drop(columns=['Valor (US$)', 'Cotação/Dia (R$)', 'Valor em R$',
                                      'Despesa no Exterior', 'Moeda'], errors='ignore')
            else:
                df = df.drop(columns=['Despesa no Exterior'], errors='ignore')

            df.insert(2, "Categoria", df.apply(lambda row: categorize(row["Lançamentos"]), axis=1))
            df["Observações"] = None

            for _, row in df.iterrows():
                transacoes.append(Transacao(
                    Data=row['Data da transação'],
                    Lançamento=row['Lançamentos'],
                    Valor=row['Valor'],
                    Categoria=row['Categoria'],
                    Observações=row.get('Observações', None)
                ))

            logging.info(f"Fatura de {mes_nome} carregada com sucesso.")
        except FileNotFoundError:
            logging.warning(f"Arquivo para {mes_nome} não encontrado.")
            continue

    df_final = pd.DataFrame([t.__dict__ for t in transacoes])
    nome_output = f"df_concatenado_categorias_{passado.strftime('%y-%m-%b').title()}.xlsx" if is_nacional else "df_concatenado_categorias.xlsx"
    output_file = pasta_mes / nome_output
    df_final.to_excel(output_file, index=False, engine='openpyxl')

    wb = load_workbook(output_file)
    ws = wb.active
    tab = Table(displayName="Transacoes", ref=ws.dimensions)
    style = TableStyleInfo(name="TableStyleMedium9", showRowStripes=True, showColumnStripes=False)
    tab.tableStyleInfo = style
    ws.add_table(tab)
    contab = NamedStyle(name="contab")
    contab.number_format = u'_-R$ * #,##0.00_-'
    for col in ws.iter_cols(min_col=1, max_col=ws.max_column):
        if col[0].value == "Data":
            for cell in col[1:]:
                cell.number_format = "DD/MM/YYYY"
        elif col[0].value == "Valor":
            for cell in col[1:]:
                cell.style = contab
    for col in ws.columns:
        try:
            ws.column_dimensions[get_column_letter(col[0].column)].width = \
                max(len(str(c.value or '')) for c in col) + 2
        except:
            pass
    wb.save(output_file)
    logging.info(f"Arquivo Excel gerado com sucesso em {output_file}.")
    return passado, data_analise, is_nacional, pasta_mes

# ==================== EXECUTAR ====================
if WIDGETS_OK:
    TIPO = tipo_widget.value
    MES = mes_widget.value
    ANO = ano_widget.value
else:
    # Fallback manual — edite aqui se os widgets não funcionarem
    TIPO = 'nacional'  # 'nacional' ou 'internacional'
    MES = datetime.now().month
    ANO = datetime.now().year

print(f"Processando: {'Nacional' if TIPO == 'nacional' else 'Internacional'} - {MES:02d}/{ANO}")
data_m_passado, data_analise, is_nacional, pasta_mes = processar(TIPO, MES, ANO)
print("Processamento concluído!")

Corrija manualmente a classificação da planilha de gastos!!!

In [ ]:
def optimize_image(input_path, output_path, max_width=800, quality=50):
    img = Image.open(input_path)
    img = img.convert("L")
    if img.width > max_width:
        ratio = max_width / img.width
        new_size = (max_width, int(img.height * ratio))
        img = img.resize(new_size, Image.LANCZOS)
    img.save(output_path, "JPEG", quality=quality, optimize=True)

if is_nacional:
    arquivo_excel = pasta_mes / f"df_concatenado_categorias_{data_m_passado.strftime('%y-%m-%b').title()}.xlsx"
else:
    arquivo_excel = pasta_mes / "df_concatenado_categorias.xlsx"

try:
    df_corrigido = pd.read_excel(arquivo_excel)
except FileNotFoundError:
    print("Arquivo df_concatenado_categorias.xlsx não encontrado.")
except PermissionError:
    print("Permissão de leitura negada para o arquivo.\nFeche o arquivo!!!")
    raise

for _, row in df_corrigido.iterrows():
    memory[row["Lançamento"].strip().upper()] = row["Categoria"]

save_category_memory(memory)

def clean_data(df_corrigido):
    df_corrigido = df_corrigido.astype({'Valor': 'float32'})
    df_corrigido_n = df_corrigido.copy()
    df_corrigido_n = df_corrigido_n.round({'Valor': 2})

    if is_nacional:
        df_corrigido['Valor'] = df_corrigido['Valor'].apply(lambda x: locale.currency(x, grouping=True))
    else:
        try:
            locale.setlocale(locale.LC_ALL, 'en_US')
        except locale.Error:
            pass
        df_corrigido['Valor'] = df_corrigido['Valor'].apply(lambda x: locale.currency(x, grouping=True))
        try:
            locale.setlocale(locale.LC_ALL, 'pt_BR.UTF-8')
        except locale.Error:
            try:
                locale.setlocale(locale.LC_ALL, 'pt_BR')
            except locale.Error:
                pass

    df_corrigido['Data'] = pd.to_datetime(df_corrigido['Data'])
    df_corrigido['Data'] = df_corrigido['Data'].dt.strftime('%d/%m/%Y')

    return df_corrigido, df_corrigido_n

df_corrigido_limpo = clean_data(df_corrigido.copy())

total = locale.currency(df_corrigido_limpo[1]['Valor'].sum(), grouping=True)

data_inicial = df_corrigido_limpo[1]['Data'].min()
data_final = df_corrigido_limpo[1]['Data'].max()

imagens = (
    df_corrigido_limpo[1][~df_corrigido_limpo[1]['Categoria'].isin(["Uber", "Cartão"])]
    .apply(lambda row: f"{row['Lançamento']} {locale.currency(row['Valor'], grouping=True)} {row['Data'].strftime('%d/%m/%Y')}", axis=1)
    .to_list()
)

def summarize_data(df_clean):
    df_clean = df_clean.groupby(['Categoria']).agg(
        Valor_sum=('Valor', 'sum'),
        Lanamentos_count=('Lançamento', 'count')
    ).reset_index()
    df_clean.insert(2, "Média", df_clean["Valor_sum"] / df_clean["Lanamentos_count"])
    df_clean = df_clean.rename(columns={'Valor_sum': 'Soma Valores', "Lanamentos_count": "Quantidade"})
    df_clean = df_clean.round({'Média': 2, 'Soma Valores': 2})
    df_clean['Soma Valores'] = df_clean['Soma Valores'].apply(lambda x: locale.currency(x, grouping=True))
    df_clean['Média'] = df_clean['Média'].apply(lambda x: locale.currency(x, grouping=True))
    return df_clean

df_resumido = summarize_data(df_corrigido_limpo[1].copy())

df = df_corrigido_limpo[0]

def load_images_from_folder(folder_path):
    image_paths = [
        os.path.abspath(os.path.join(folder_path, f)).replace("\\", "/")
        for f in os.listdir(folder_path)
        if f.endswith(('.png', '.jpg', '.jpeg'))
    ]
    return image_paths

image_folder_path = str(pasta_mes / "NF").replace("\\", "/")

optimized_folder = os.path.join(image_folder_path, "optimized")
os.makedirs(optimized_folder, exist_ok=True)

image_paths = []
for img_path in load_images_from_folder(image_folder_path):
    filename = os.path.basename(img_path)
    new_path = os.path.join(optimized_folder, filename.replace(".png", ".jpg"))
    optimize_image(img_path, new_path)
    image_paths.append(new_path.replace("\\", "/"))

def generate_pdf(output_file=None, template_name='template.html', context=None):
    context = contexto or {}
    templates_dir = r'C:\Users\vital\Documents\programas\Relatorio_Financeiro_Mensal'
    env = Environment(loader=FileSystemLoader(templates_dir))
    env.globals['zip'] = zip
    template = env.get_template(template_name)
    html_content = template.render(context)
    temp_html_path = os.path.join(templates_dir, 'temp_rendered.html')
    with open(temp_html_path, 'w', encoding='utf-8') as f:
        f.write(html_content)
    os.makedirs(str(pasta_mes), exist_ok=True)
    pdf_path = os.path.join(str(pasta_mes), output_file or f"Relatório de Transações de {data_m_passado.strftime('%B %Y').title()}.pdf")
    html = HTML(filename=temp_html_path)
    css = CSS(filename='styles.css')
    html.write_pdf(pdf_path, stylesheets=[css])
    print(f"PDF gerado em: '{pdf_path}'")
    os.remove(temp_html_path)

contexto = {
    "hoje": data_analise.strftime('%Y-%m-%dT%H:%M:%S%:z'),
    "data": df.values.tolist(),
    "data_a": df_resumido.values.tolist(),
    "data_inicial": data_inicial.strftime('%d/%m/%Y'),
    "data_final": data_final.strftime('%d/%m/%Y'),
    "columns": df.columns.tolist(),
    "columns_a": df_resumido.columns.tolist(),
    "images": image_paths,
    "images_names": imagens,
    "datas": data_m_passado.strftime('%B %Y').title(),
    "total": total,
}

generate_pdf()

In [ ]:
from pathlib import Path
from pypdf import PdfReader, PdfWriter
import os

relatorio_principal = pasta_mes / f"Relatório de Transações de {data_m_passado.strftime('%B %Y').title()}.pdf"
pasta_urbi = pasta_mes / "Urbi"
arquivo_saida = pasta_mes / f"Relatório de Transações e Uber de {data_m_passado.strftime('%B %Y').title()}.pdf"

if not relatorio_principal.exists():
    print(f"Relatório principal não encontrado: {relatorio_principal}")
elif not pasta_urbi.exists():
    print(f"Pasta Urbi não encontrada: {pasta_urbi}")
else:
    writer = PdfWriter()

    print(f"Adicionando: {relatorio_principal.name}")
    reader = PdfReader(str(relatorio_principal))
    for pagina in reader.pages:
        writer.add_page(pagina)

    pdfs_urbi = sorted(pasta_urbi.glob("*.pdf"))
    for pdf in pdfs_urbi:
        print(f"Adicionando: {pdf.name}")
        reader = PdfReader(str(pdf))
        for pagina in reader.pages:
            writer.add_page(pagina)

    with open(arquivo_saida, "wb") as f:
        writer.write(f)

    tamanho_mb = os.path.getsize(arquivo_saida) / (1024 * 1024)
    print("\n===================================")
    print("PDF final criado com sucesso!")
    print(f"Arquivo: {arquivo_saida.name}")
    print(f"Tamanho: {tamanho_mb:.2f} MB")
    print("===================================")
